# Processing Status Dashboard

Shows which tiles have been processed by reading:
- `tile_list.json` — the full set of land tiles (static, created by notebook 01)
- Icechunk commit history — the single source of truth for processed tiles

Set the four Azure env vars before running:
```
AZURE_STORAGE_ACCOUNT, AZURE_STORAGE_SAS_TOKEN, AZURE_CONTAINER, ICECHUNK_PREFIX
```

In [ ]:
import re
import sys
from pathlib import Path

import icechunk
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from utils import get_storage, load_config, load_tile_list

cfg = load_config()
TILE_ROWS = cfg["TILE_ROWS"]
TILE_COLS = cfg["TILE_COLS"]

## Load tile list and check Icechunk history

In [ ]:
land_tiles = load_tile_list()
land_set = {(t["row"], t["col"]) for t in land_tiles}

storage = get_storage()
repo = icechunk.Repository.open(storage)

pattern = re.compile(r"tile_(\d+)_(\d+): processed")
processed = set()
for commit in repo.ancestry(branch="main"):
    m = pattern.match(commit.message)
    if m:
        processed.add((int(m.group(1)), int(m.group(2))))

remaining = land_set - processed

print(f"Land tiles:  {len(land_set)}")
print(f"Processed:   {len(processed)}")
print(f"Remaining:   {len(remaining)}")
print(f"Progress:    {100 * len(processed) / len(land_set):.1f}%")

## Visualize tile grid

In [ ]:
# 0 = ocean, 1 = unprocessed land, 2 = processed
grid = np.zeros((TILE_ROWS, TILE_COLS), dtype=np.uint8)
for r, c in land_set:
    grid[TILE_ROWS - 1 - r, c] = 1  # flip rows so north is up
for r, c in processed:
    grid[TILE_ROWS - 1 - r, c] = 2

fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.matplotlib.colors.ListedColormap(["#d0e8f5", "#f5a623", "#4caf50"])
ax.imshow(grid, cmap=cmap, vmin=0, vmax=2, aspect="auto",
          extent=[-180, 180, -90, 90])

ax.set_xticks(range(-180, 181, 30))
ax.set_yticks(range(-90, 91, 30))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"Tile processing status — {len(processed)}/{len(land_set)} done "
    f"({100*len(processed)/len(land_set):.1f}%)"
)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#d0e8f5", label="ocean"),
    Patch(color="#f5a623", label="unprocessed"),
    Patch(color="#4caf50", label="processed"),
], loc="lower left")

plt.tight_layout()
plt.show()

## List unprocessed tiles

In [ ]:
if remaining:
    print("Unprocessed tiles (row, col):")
    for r, c in sorted(remaining):
        lat_min = -90 + r * cfg["TILE_SIZE_DEG"]
        lon_min = -180 + c * cfg["TILE_SIZE_DEG"]
        print(f"  row={r}, col={c}  lat=[{lat_min:.0f}, {lat_min+10:.0f}]  lon=[{lon_min:.0f}, {lon_min+10:.0f}]")
else:
    print("All land tiles processed.")